# 1. Generate and understand the optical telemetry
Run `python -m pip install -e ".[dev]"` from the repository root first.
One configuration describes the simulator, feature settings and incident candidates.
The simulator is a controlled test fixture, not a calibrated network digital twin.
Ground truth stays in a separate file. EDA below reads only the training interval.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
# Resolve relative configured output paths consistently from any notebook.
import os
os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [start + pd.Timedelta(days=settings["generator"]["days"] * f)
              for f in settings["splits"]]


In [ ]:
native = pd.read_parquet(
    RUN / "telemetry.parquet", filters=[("time", "<", boundaries[0])]
)
display(native.describe(include="number"))
display(native.groupby("device")[["rx_dbm", "ber"]].apply(lambda x: x.isna().mean()).head())
entity = native.device.iloc[0]
view = native.loc[native.device.eq(entity)].set_index("time")
view.rx_dbm.plot(figsize=(12, 3), ylabel="Rx power (dBm)", title=entity)
plt.show()
centred = native.assign(residual=native.rx_dbm - native.groupby("device").rx_dbm.transform("median"))
centred.groupby(centred.time.dt.hour).residual.median().plot(
    ylabel="Centred power (dB)", title="Daily profile; differing device phases can cancel"
)
plt.show()


Physics and measurement are separate. Stationary Gaussian AR(1) noise has a
known variance and correlation time. Daily variation is deterministic per device.
Faults are drifted random walks, accelerating attenuation or variance shifts.
BER is an illustrative monotone margin response; it is retained for EDA, not used
as independent evidence by the detector. Onset, observable-onset proxy and impact
are separate labels. A variance shift need not cause low-power impact.
The first 55% is deliberately fault-free: a baseline assumption to challenge later.